# Noise Classification — Evaluation Metrics
eval 결과 파일을 읽어서 Accuracy, F1, mAP 등 평가 지표를 계산합니다.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, average_precision_score
)
from sklearn.preprocessing import label_binarize

LABEL_LIST = [
    'clean',
    'background_noise',
    'background_music',
    'gaussian_noise',
    'band_pass_filter',
    'echo',
    'pitch_shift',
    'time_stretch',
    'reverberation',
    'auto_tune',
]

## 설정 — 평가할 결과 파일 경로를 여기서 변경

In [20]:
# ── 평가 결과 파일 경로 ──────────────────────────────────────────────────────
# train.py --is_eval 로 생성된 파일 (탭 구분, 헤더 있음)
# 형식: file_path  true_label  predicted_label  score  clean  background_noise  ...
EVAL_RESULT_PATH = 'results/in-domain/hubert_eval.txt'

# 모델 이름 (그래프 제목용)
MODEL_NAME = 'SSAST-Base-Patch-400'

## 데이터 로드

In [21]:
df = pd.read_csv(EVAL_RESULT_PATH, sep='\t')
print(f'총 샘플 수: {len(df)}')
df.head()

총 샘플 수: 14948


,file_path,true_label,predicted_label,score,clean,background_noise,background_music,gaussian_noise,band_pass_filter,echo,pitch_shift,time_stretch,reverberation,auto_tune
0,/nvme3/Datasets/WJ/noise_dataset/NC/eval/auto_...,auto_tune,auto_tune,0.9978,0.0007,0.0002,0.0002,0.0000,0.0,0.0008,0.0001,0.0001,0.0001,0.9978
1,/nvme3/Datasets/WJ/noise_dataset/NC/eval/auto_...,auto_tune,auto_tune,0.9960,0.0012,0.0003,0.0003,0.0001,0.0,0.0017,0.0002,0.0001,0.0001,0.9960
2,/nvme3/Datasets/WJ/noise_dataset/NC/eval/auto_...,auto_tune,auto_tune,0.9979,0.0007,0.0002,0.0001,0.0000,0.0,0.0007,0.0001,0.0001,0.0001,0.9979
3,/nvme3/Datasets/WJ/noise_dataset/NC/eval/auto_...,auto_tune,auto_tune,0.9980,0.0007,0.0002,0.0001,0.0000,0.0,0.0006,0.0001,0.0001,0.0001,0.9980
4,/nvme3/Datasets/WJ/noise_dataset/NC/eval/auto_...,auto_tune,auto_tune,0.9980,0.0008,0.0002,0.0001,0.0000,0.0,0.0007,0.0001,0.0001,0.0001,0.9980


## 전체 지표 — Accuracy / F1 / mAP

In [22]:
y_true = df['true_label'].tolist()
y_pred = df['predicted_label'].tolist()

# ── Accuracy ──────────────────────────────────────────────────────────────────
acc = accuracy_score(y_true, y_pred)

# ── F1 ────────────────────────────────────────────────────────────────────────
f1_macro    = f1_score(y_true, y_pred, average='macro',    zero_division=0)
f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)

# ── mAP (mean Average Precision) ──────────────────────────────────────────────
# 각 클래스별로 one-vs-rest AP를 계산한 뒤 평균
# 결과 파일의 클래스별 softmax 확률 컬럼을 사용
prob_cols = [c for c in LABEL_LIST if c in df.columns]

if prob_cols:
    y_true_bin = label_binarize(y_true, classes=LABEL_LIST)   # (N, 10)
    y_scores   = df[LABEL_LIST].values                         # (N, 10)

    # 클래스별 AP
    per_class_ap = [
        average_precision_score(y_true_bin[:, i], y_scores[:, i])
        for i in range(len(LABEL_LIST))
    ]
    mAP = np.mean(per_class_ap)
else:
    mAP = float('nan')
    per_class_ap = [float('nan')] * len(LABEL_LIST)
    print('[경고] 클래스별 확률 컬럼이 없습니다. mAP를 계산할 수 없습니다.')

print(f'Model        : {MODEL_NAME}')
print(f'Accuracy     : {acc*100:.2f}%')
print(f'F1 (macro)   : {f1_macro:.4f}')
print(f'F1 (weighted): {f1_weighted:.4f}')
print(f'mAP          : {mAP:.4f}')

Model        : SSAST-Base-Patch-400
Accuracy     : 97.17%
F1 (macro)   : 0.9721
F1 (weighted): 0.9719
mAP          : 0.9871


## 클래스별 리포트

In [ ]:
report = classification_report(
    y_true, y_pred,
    labels=LABEL_LIST,
    zero_division=0
)
print(report)

## 클래스별 Accuracy / AP 비교

In [ ]:
per_class = (
    df.assign(correct=df['true_label'] == df['predicted_label'])
      .groupby('true_label')['correct']
      .agg(['sum', 'count'])
      .rename(columns={'sum': 'correct', 'count': 'total'})
)
per_class['accuracy'] = per_class['correct'] / per_class['total']
per_class['AP']       = pd.Series(dict(zip(LABEL_LIST, per_class_ap)))
per_class = per_class.reindex(LABEL_LIST)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Accuracy
ax = axes[0]
bars = ax.bar(per_class.index, per_class['accuracy'] * 100, color='steelblue', edgecolor='white')
ax.axhline(acc * 100, color='tomato', linestyle='--', label=f'Overall {acc*100:.1f}%')
ax.set_ylim(0, 115)
ax.set_title(f'{MODEL_NAME} — Per-class Accuracy')
ax.set_ylabel('Accuracy (%)')
ax.legend()
for bar, val in zip(bars, per_class['accuracy']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val*100:.1f}', ha='center', va='bottom', fontsize=8)
ax.tick_params(axis='x', rotation=35)

# AP
ax = axes[1]
bars = ax.bar(per_class.index, per_class['AP'], color='seagreen', edgecolor='white')
ax.axhline(mAP, color='tomato', linestyle='--', label=f'mAP {mAP:.3f}')
ax.set_ylim(0, 1.15)
ax.set_title(f'{MODEL_NAME} — Per-class AP')
ax.set_ylabel('Average Precision')
ax.legend()
for bar, val in zip(bars, per_class['AP']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8)
ax.tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()

print(per_class[['correct', 'total', 'accuracy', 'AP']].to_string())

## 혼동행렬 (Confusion Matrix)

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=LABEL_LIST)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=LABEL_LIST, yticklabels=LABEL_LIST,
    linewidths=0.5, ax=ax
)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title(f'{MODEL_NAME} — Confusion Matrix', fontsize=13)
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 정규화 혼동행렬 (Recall 기준)

In [ ]:
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=LABEL_LIST, yticklabels=LABEL_LIST,
    vmin=0, vmax=1, linewidths=0.5, ax=ax
)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title(f'{MODEL_NAME} — Normalized Confusion Matrix (Recall)', fontsize=13)
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 모델 간 비교 (여러 결과 파일 한꺼번에)

In [ ]:
# 비교할 모델 결과 파일 목록
# 형식: { '모델 이름': '결과 파일 경로' }
COMPARE_FILES = {
    'SSAST-Base':  'results/in-domain/ssast_base_patch_400_eval.txt',
    'SSAST-Small': 'results/in-domain/ssast_small_patch_400_eval.txt',
    'SSAST-Tiny':  'results/in-domain/ssast_tiny_patch_400_eval.txt',
    'HTSAT':       'results/in-domain/htsat_eval.txt',
    'HuBERT':      'results/in-domain/hubert_eval.txt',
    'Wav2Vec2':    'results/in-domain/wav2vec2_eval.txt',
}

def compute_metrics(path):
    d   = pd.read_csv(path, sep='\t')
    yt  = d['true_label']
    yp  = d['predicted_label']
    acc = accuracy_score(yt, yp)
    f1  = f1_score(yt, yp, average='macro', zero_division=0)
    prob_cols = [c for c in LABEL_LIST if c in d.columns]
    if prob_cols:
        yb = label_binarize(yt, classes=LABEL_LIST)
        ys = d[LABEL_LIST].values
        map_ = np.mean([average_precision_score(yb[:, i], ys[:, i]) for i in range(len(LABEL_LIST))])
    else:
        map_ = float('nan')
    return {'accuracy': acc * 100, 'f1_macro': f1, 'mAP': map_}

rows = []
for name, path in COMPARE_FILES.items():
    try:
        rows.append({'model': name, **compute_metrics(path)})
    except FileNotFoundError:
        print(f'[skip] {path} 파일 없음')

if rows:
    compare_df = pd.DataFrame(rows).set_index('model')
    display(compare_df.round(4))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    metrics = [('accuracy', 'Accuracy (%)'), ('f1_macro', 'F1 Macro'), ('mAP', 'mAP')]
    for ax, (col, title) in zip(axes, metrics):
        compare_df[col].plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
        ax.set_title(title, fontsize=12)
        ax.set_xlabel('')
        ax.tick_params(axis='x', rotation=30)
        for p in ax.patches:
            ax.annotate(f'{p.get_height():.2f}',
                        (p.get_x() + p.get_width()/2, p.get_height()),
                        ha='center', va='bottom', fontsize=9)
    plt.suptitle('Model Comparison — In-domain', fontsize=13)
    plt.tight_layout()
    plt.show()